# 04 — Évaluation
CLIP-score (respect du texte), similarité au domaine, mémorisation (DINOv2), diversité, coût.

In [ ]:
# Setup — works on Colab / Kaggle AND locally (VS Code, Jupyter)
import os, sys, subprocess
from pathlib import Path

ON_CLOUD = "google.colab" in sys.modules or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if ON_CLOUD:
    if not Path("lora-zellige").exists():
        subprocess.run(["git", "clone", "https://github.com/braziyass/lora-zellige.git"], check=True)
    os.chdir("lora-zellige")
else:
    # Locally the notebook starts in notebooks/ : go up to the repo root
    root = Path.cwd().resolve()
    while not (root / "requirements.txt").exists() and root != root.parent:
        root = root.parent
    os.chdir(root)
sys.path.insert(0, os.getcwd())
PY = f'"{sys.executable}"'   # used as !{PY} -m ... in the cells below
print("Working directory:", os.getcwd())

print("Python used by this notebook:", sys.executable)

# Packages: installed automatically on Colab/Kaggle only.
# Locally, install once in the Terminal (see README), or set INSTALL = True.
INSTALL = ON_CLOUD
if INSTALL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (fine for notebooks 01-02 data steps)")
except ImportError:
    print("torch not installed: only needed from notebook 02 on (use Colab/Kaggle for training)")

In [ ]:
!{PY} -m src.generate --config configs/base.yaml --include_base

In [ ]:
!{PY} -m src.evaluate --config configs/base.yaml

In [ ]:
import pandas as pd
s = pd.read_csv("results/metrics_summary.csv")
s[s.category.isin(["composition", "all"])].sort_values(["category", "rank", "step"])

In [ ]:
from IPython.display import Image, display
for f in ["curves_composition", "curves_in_domain", "tradeoff", "grid_c01_s0"]:
    display(Image(f"results/figures/{f}.png"))

## Intensité de la LoRA sur la meilleure configuration

In [ ]:
BEST = "outputs/loras/r8/step_1000"   # ← à remplacer par la meilleure config
for sc in [0.5, 0.8, 1.2]:
    !{PY} -m src.generate --config configs/base.yaml --lora {BEST} --scale {sc}
!{PY} -m src.evaluate --config configs/base.yaml

## Quasi-copies détectées

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image as PImage
p = pd.read_csv("results/metrics_per_image.csv").sort_values("train_nn_sim", ascending=False).head(6)
fig, axes = plt.subplots(1, 6, figsize=(18, 3.5))
for ax, (_, r) in zip(axes, p.iterrows()):
    ax.imshow(PImage.open(r.file)); ax.axis("off")
    ax.set_title(f"{r.config}\nsim={r.train_nn_sim:.2f}", fontsize=7)
plt.savefig("results/figures/top_copies.png", dpi=150)